In [ ]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set. Add it to your .env file.")

genai.configure(api_key=gemini_api_key)
gemini_client = genai.GenerativeModel("gemini-2.5-flash")

print("API client successfully configured")


In [ ]:
from IPython.display import display, Markdown

def print_markdown(text):
    display(Markdown(text))

In [ ]:
def get_ai_tutor_response(user_question):
    """
    Sends a question to the Gemini API, asking it to respond as an AI Tutor.
    Args:
        user_question (str): The question asked by the user
    Returns:
        str: The AI's response, or an error message
    """
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        response = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            )
        )
        return response.text
    except Exception as e:
        return f"Sorry, I encountered an error trying to get an answer: {e}"


In [ ]:
test_question = "Could you explain the concept of functions in Python and their purpose in programming?"

tutor_answer = get_ai_tutor_response(test_question)

print_markdown(f"""
### Asking the AI Tutor

{test_question}

### AI Tutor's Response

{tutor_answer}
""")

In [ ]:
import gradio as gr

In [ ]:
ai_tutor_interface_simple = gr.Interface(
    fn = get_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder= "Ask the AI Tutor anything...", label= "Your Question"),
    outputs= gr.Textbox(label = "AI Tutor's Answer"),
    title= "🤖 Simple AI Tutor",
    description= "Enter your question below and the AI Tutor will provide an explanation. Powered by GeminiAI.",
    flagging_mode = "never",
)
print("Launching Gradio Interface...")
ai_tutor_interface_simple.launch()

In [ ]:
def stream_ai_tutor_response(user_question):
    """
    Sends a question to the GeminiAI API and streams the response as a generator.
    Args:
        user_question (str): The question asked by the user.
    Yields:
        str: Chunks of the AI's response. 
    """
    system_prompt = "You are a helpful and patient AI Tutor. Explain concepts clearly and concisely."

    try:
        stream = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            ),
            stream=True
        )
        full_response = ""
        for chunk in stream:
            if chunk.text:
                text_chunk = chunk.text
                full_response += text_chunk
                yield full_response
    
    except Exception as e:
        yield f"Sorry, I encountered an error: {e}"


In [ ]:
ai_tutor_interface_streaming = gr.Interface(
    fn = stream_ai_tutor_response,
    inputs = gr.Textbox(lines = 2, placeholder= "Ask the AI Tutor anything...", label= "Your Question"),
    outputs= gr.Markdown(
        label = "AI Tutor's Answer (Streaming)", container= True, height= 250
    ),
    title= "🤖 AI Tutor with Streaming",
    description= "Enter your question. The answer will appear word-by-word!",
    flagging_mode = "never",
)
print("Launching Streaming Gradio Interface...")
ai_tutor_interface_streaming.launch()

In [ ]:
explanation_levels= {
    1: "like I'm 5 years old",
    2: "Like I'm 10 years old",
    3: "like a high school student",
    4: "like a college student",
    5: "like an expert in the field"
}

In [ ]:
def stream_ai_tutor_response_with_level(user_question, explanation_level_value):
    """
    Streams AI Tutor response based on user question and selected explanation level.
    Args:
        user_question (str): The question from the user.
        explanation_level_value (int): The value from the slider (1-5).
    
    Yields:
        str: Chunks of the AI's response.
    """
    level_description = explanation_levels.get(
        explanation_level_value, "clearly and concisely"
    )
    system_prompt = f"You are a helpful AI Tutor. Explain the following concept {level_description}"
    print(f"DEBUG: Using System Prompt: '{system_prompt}'")

    try:
        stream = gemini_client.generate_content(
            f"{system_prompt}\n\nStudent question: {user_question}",
            generation_config=genai.types.GenerationConfig(
                temperature=0.7
            ),
            stream=True
        )
        full_response = ""

        for chunk in stream:
            if chunk.text:
                text_chunk = chunk.text
                full_response += text_chunk
                yield full_response
    except Exception as e:
        print(f"An error occurred during streaming: {e}")
        yield f"Sorry, I encountered an error: {e}"

In [ ]:
ai_tutor_interface_slider = gr.Interface(
    fn= stream_ai_tutor_response_with_level,
    inputs=[
        gr.Textbox(lines = 3, placeholder= "Ask the AI Tutor a question...", label = "Your Question"),
        gr.Slider(
            minimum= 1,
            maximum= 5,
            step = 1,
            value= 3, #default level (high school)
            label= "Explanation Level",
        ),
    ],
    outputs= gr.Markdown(label="AI Tutor's Explanation (Streaming)", container= True, height= 250),
    title= "🎓 Advanced AI Tutor",
    description= "Ask a question and select and desired level of explanation using the slider.",
    flagging_mode = "never",
)
print("Launching Advanced Interface with Slider...")
ai_tutor_interface_slider.launch()